# 🛡️ 01. Exploratory Data Analysis (EDA) — PE Malware Detection
**Project**: XGBoost-Powered PE Malware Detection with Interaction Constraints  
**Dataset**: 50,000 synthetic PE executable samples (EMBER-inspired schema)  
**Author**: Aaryan Bangale


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

data_path = Path('../data/synthetic_malware_data.csv')
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()


## 1. Class Distribution Analysis
Malware detection datasets typically suffer from natural class imbalance. Here we examine the benign vs malware ratio.


In [ ]:
class_counts = df['label'].value_counts()
class_pcts = df['label'].value_counts(normalize=True) * 100

print(f"Benign (0):  {class_counts[0]:,} ({class_pcts[0]:.2f}%)")
print(f"Malware (1): {class_counts[1]:,} ({class_pcts[1]:.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='label', data=df, palette=['#00f5d4', '#ff007f'], ax=ax[0])
ax[0].set_title('Class Distribution (Count)')
ax[0].set_xticklabels(['Benign (0)', 'Malware (1)'])

ax[1].pie(class_counts, labels=['Benign', 'Malware'], autopct='%1.1f%%', 
         colors=['#00f5d4', '#ff007f'], startangle=140, explode=[0, 0.08])
ax[1].set_title('Class Distribution (Percentage)')
plt.tight_layout()
plt.show()


## 2. Feature Groups & Descriptive Statistics
The dataset contains 33 features categorized into 6 logical domain groups:
- **PE Header**: file_size, num_sections, entry_point_offset, image_base, has_debug, has_signature, dll_characteristics
- **Section Entropy**: avg_section_entropy, max_section_entropy, section_size_variance, num_executable_sections, num_writable_sections
- **Import APIs**: num_imports, num_suspicious_imports, num_unique_dlls, uses_crypto_api, uses_network_api, uses_process_api, uses_registry_api
- **String Features**: num_urls, num_ips, num_registry_keys, num_file_paths, avg_string_length, num_printable_strings
- **Behavioral Indicators**: polymorphic_score, obfuscation_level, code_mutation_rate, api_call_frequency, memory_allocation_pattern
- **Network Telemetry**: packet_entropy, connection_attempts, c2_similarity_score


In [ ]:
df.describe().T[['mean', 'std', 'min', '50%', 'max']]


## 3. Correlation with Malware Label
Let's inspect which features exhibit the strongest linear and non-linear correlation with the malware target label.


In [ ]:
corr = df.corr()
label_corr = corr['label'].drop('label').sort_values(ascending=False)

plt.figure(figsize=(12, 8))
colors = ['#ff007f' if c > 0 else '#00f5d4' for c in label_corr.values]
label_corr.plot(kind='barh', color=colors)
plt.title('Feature Correlation with Target Label (Malware)', fontsize=14)
plt.xlabel('Pearson Correlation Coefficient')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Key Feature Distributions (Benign vs Malware)
Packed and polymorphic malware samples typically exhibit higher entropy and suspicious import counts.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.kdeplot(data=df, x='max_section_entropy', hue='label', common_norm=False, fill=True, palette=['#00f5d4', '#ff007f'], ax=axes[0, 0])
axes[0, 0].set_title('Max Section Entropy Distribution')

sns.boxplot(data=df, x='label', y='polymorphic_score', palette=['#00f5d4', '#ff007f'], ax=axes[0, 1])
axes[0, 1].set_title('Polymorphic Score by Class')
axes[0, 1].set_xticklabels(['Benign', 'Malware'])

sns.kdeplot(data=df, x='c2_similarity_score', hue='label', common_norm=False, fill=True, palette=['#00f5d4', '#ff007f'], ax=axes[1, 0])
axes[1, 0].set_title('C2 Similarity Score Distribution')

sns.boxplot(data=df, x='label', y='num_suspicious_imports', palette=['#00f5d4', '#ff007f'], ax=axes[1, 1])
axes[1, 1].set_title('Suspicious Imports Count by Class')
axes[1, 1].set_xticklabels(['Benign', 'Malware'])

plt.tight_layout()
plt.show()
